# 🛡️ Elastic Detection Rules — TOML ➜ Single Kibana NDJSON File

**Convert an entire Elastic detection-rules category into ONE combined `.ndjson` file, ready to import into Kibana
in a single upload.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
![Python](https://img.shields.io/badge/python-3.12%2B-blue)
![License](https://img.shields.io/badge/license-MIT-green)

> This is the **single-file variant** of the converter. If you'd rather have one `.ndjson` per individual rule
> (zipped), use `Elastic_Detection_Rules_Converter.ipynb` in this repo instead. If you want a standalone
> command-line tool / `.exe` with no notebook at all, see `Elastic_Detection_Rules_Converter_EXE.ipynb`.

> ⭐ If this notebook saved you time, please **star** the repo — it helps other security engineers find it too!


## 📑 Table of Contents

1. [Overview](#overview)
2. [How It Works](#how-it-works)
3. [Prerequisites](#prerequisites)
4. [Step 1 — Install Python Dependencies](#step-1)
5. [Step 2 — Clone the Elastic Detection Rules Repository](#step-2)
6. [Step 3 — Install Poetry](#step-3)
7. [Step 4 — Install Project Dependencies](#step-4)
8. [Step 5 — Verify the CLI](#step-5)
9. [Step 6 — Choose the Category](#step-6)
10. [Step 7 — Export the Category into One Combined NDJSON](#step-7)
11. [Step 8 — Verify the Output](#step-8)
12. [Step 9 — Download the NDJSON File](#step-9)
13. [Importing into Kibana](#importing)
14. [References](#references)
15. [License & Disclaimer](#license)


<a id="overview"></a>
## 📖 Overview

### Why a single-file variant?

The [`elastic/detection-rules`](https://github.com/elastic/detection-rules) repository stores every detection
rule as its own `.toml` file. Kibana's Security app only imports `.ndjson`, and while you *can* import many
individual `.ndjson` files at once, it's often far more convenient to hand Kibana **one file containing every
rule in a category** — a single drag-and-drop, a single import confirmation, done.

This notebook does exactly that: it exports an entire rule category (e.g. every Windows rule) into **one
combined `.ndjson` file**, using the `-d` (directory) mode of Elastic's own `export-rules-from-repo` CLI command
— the same officially supported tool, just pointed at a whole folder instead of one file at a time.

### When to use this vs. the per-rule notebook

| | This notebook (single file) | `Elastic_Detection_Rules_Converter.ipynb` (per-rule) |
|---|---|---|
| Output | One `.ndjson` containing every rule in the category | One `.ndjson` per individual rule, zipped |
| Best for | Fast, one-shot bulk import of an entire category | Cherry-picking specific rules, reviewing rules individually, granular version control |
| Import steps in Kibana | 1 drag-and-drop | 1 drag-and-drop of many files (or unzip first) |


<a id="how-it-works"></a>
## ⚙️ How It Works

```
elastic/detection-rules (GitHub)
        │  git clone
        ▼
   rules/<category>/*.toml            (hundreds of individual rule files)
        │  poetry run python -m detection_rules
        │      export-rules-from-repo -d <category_dir> -o <combined>.ndjson
        ▼
   <category>_rules.ndjson             (ONE file, every rule in the category)
        │  Stack Management → Rules → Import rules
        ▼
   Rules loaded into Kibana Security app
```


<a id="prerequisites"></a>
## ✅ Prerequisites

- Runs out-of-the-box on **Google Colab** (recommended) or any Linux/macOS machine with Python 3.12+, `git`, and
  internet access to `github.com` and `pypi.org`.
- No Elastic Cloud / Kibana credentials are required — this notebook only performs a local, offline file
  conversion. You only need Kibana access later, to import the resulting file.


<a id="step-1"></a>
## 1️⃣ Install Python Dependencies


In [ ]:
!pip install -q requests toml

<a id="step-2"></a>
## 2️⃣ Clone the Elastic Detection Rules Repository

Pulls the **latest** version of Elastic's official rule set and CLI tooling directly from GitHub.


In [ ]:
!git clone --depth 1 https://github.com/elastic/detection-rules.git

### Change into the Repository Directory

Using the `%cd` magic (rather than a plain `cd`) so the working directory change reliably persists across every
later cell in the notebook.


In [ ]:
%cd detection-rules

<a id="step-3"></a>
## 3️⃣ Install Poetry

[Poetry](https://python-poetry.org/) is the dependency manager the `detection-rules` project uses to pin and
install its exact Python dependencies.


In [ ]:
!pip install -q poetry

<a id="step-4"></a>
## 4️⃣ Install Project Dependencies

Reads `pyproject.toml` / `poetry.lock` and installs the exact dependency versions the `detection_rules` CLI was
built and tested against. Can take a couple of minutes the first time.


In [ ]:
!poetry install

<a id="step-5"></a>
## 5️⃣ Verify the CLI Installed Correctly

If this prints the `detection_rules` help/usage banner, the environment is ready.


In [ ]:
!poetry run python -m detection_rules --help

<a id="step-6"></a>
## 6️⃣ Choose the Category

Elastic organizes rules by platform under `rules/` (e.g. `windows`, `linux`, `macos`, `network`, `cloud`). List
the available categories, then set `RULE_CATEGORY` below to the one you want — or use `"rules"` itself to combine
**every** platform into one file.


In [ ]:
import os

RULES_ROOT = "rules"
print("Available rule categories:\n")
for entry in sorted(os.listdir(RULES_ROOT)):
    full_path = os.path.join(RULES_ROOT, entry)
    if os.path.isdir(full_path):
        print(f" - {entry}")


In [ ]:
# Set this to any category printed above, or "rules" itself to combine every platform into one file.
RULE_CATEGORY = "windows"

RULES_DIR = os.path.join(RULES_ROOT, RULE_CATEGORY) if RULE_CATEGORY != "rules" else RULES_ROOT
OUTPUT_FILE = f"{RULE_CATEGORY}_rules.ndjson"

print(f"Source directory : {RULES_DIR}")
print(f"Output file      : {OUTPUT_FILE}")


<a id="step-7"></a>
## 7️⃣ Export the Category into One Combined NDJSON

Calls `export-rules-from-repo` with `-d` (recursively read every `.toml` in the directory) and `-o` (write a
single combined output file) — this is Elastic's own documented bulk-export mode.


In [ ]:
import subprocess

result = subprocess.run(
    [
        "poetry", "run", "python", "-m", "detection_rules",
        "export-rules-from-repo",
        "-d", RULES_DIR,
        "-o", OUTPUT_FILE,
    ],
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("❌ Export failed:")
    print(result.stderr)
else:
    print(f"✅ Export succeeded → {OUTPUT_FILE}")


<a id="step-8"></a>
## 8️⃣ Verify the Output

Each line in an `.ndjson` file is one JSON rule object, so counting lines tells you how many rules made it into
the file. This also previews the first rule so you can eyeball the structure.


In [ ]:
with open(OUTPUT_FILE, "r") as f:
    lines = f.readlines()

print(f"{OUTPUT_FILE} contains {len(lines)} rule(s).\n")
print("Preview of the first rule:\n")
print(lines[0][:500] + ("..." if len(lines[0]) > 500 else ""))


<a id="step-9"></a>
## 9️⃣ Download the NDJSON File

Downloads the single combined `.ndjson` file directly — no zipping needed, since there's only one file. This
cell only works inside **Google Colab**; if running locally, the file is already saved at `OUTPUT_FILE` on disk.


In [ ]:
try:
    from google.colab import files
    files.download(OUTPUT_FILE)
except ImportError:
    print("Not running in Google Colab — grab the file directly from disk instead:")
    print(f"  {OUTPUT_FILE}")


<a id="importing"></a>
## 📥 Importing the NDJSON into Kibana

1. Open **Kibana** → **Security** → **Manage** → **Rules** → **Detection rules (SIEM)**.
2. Click **Import rules**.
3. Drag and drop the single `.ndjson` file downloaded in Step 9.
4. Optionally enable:
   - **Overwrite existing detection rules with conflicting `rule_id`**
   - **Overwrite existing exception lists with conflicting `list_id`**
5. Click **Import** and confirm the rule count matches the count printed in Step 8.


<a id="references"></a>
## 📚 References

- Elastic Detection Rules repository: https://github.com/elastic/detection-rules
- CLI reference (`export-rules-from-repo`): https://github.com/elastic/detection-rules/blob/main/CLI.md
- Elastic docs — Manage detection rules (import/export): https://www.elastic.co/docs/solutions/security/detect-and-alert/manage-detection-rules


## 🤝 Contributing & ⭐ Support

If this notebook was useful, please consider **starring** the repository and **forking** it to adapt for your
own environment. Issues and PRs are welcome!


<a id="license"></a>
## ⚖️ License & Disclaimer

This notebook is an independent automation wrapper and is **not officially affiliated with or endorsed by
Elastic**. It automates Elastic's own publicly documented CLI commands from
[`elastic/detection-rules`](https://github.com/elastic/detection-rules), which is separately licensed by Elastic.

Suggested license for this wrapper notebook/repository: MIT.
